In [1]:
!pip install -U llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 22.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.35-py3-none-linux_x86_64.whl size=20848225 sha256=361cf083cebc8dd720171327059874a8bec20c2d487e34cf3b98b935ecdcd6e6
  Stored in directory: /root/.cache/pip/wheels/1b/64/d4/17744d793e69b485a7664ef47b18e402a72a6e08e84f7b9926
Successfully built llama-cpp-python


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/wangapa106g/massage-analysis/test_50_consistent.csv


In [3]:

from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="EngineerWanga0791709020/SME-Ledger",
	filename="sme-ledger-v2-Q4_K_M.gguf",
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./sme-ledger-v2-Q4_K_M.gguf:   0%|          | 0.00/261M [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 33 key-value pairs and 236 tensors from /root/.cache/huggingface/hub/models--EngineerWanga0791709020--SME-Ledger/snapshots/1aa4d6566c59f07727d7a814269122c8f037dd09/./sme-ledger-v2-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 64
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                               general.name str              = Merged
llama_model_loader: - kv   5:                         general.size_label str              = 268M
llama_model_loader: - kv   6:                       

In [4]:
# ============================================================
# SME-LEDGER V2 — MANUAL 20-SAMPLE EVALUATION
#
# You manually judge:
#   - JSON validity
#   - Correctness
#   - Missing information handling
#   - Capability answers
#
# This script ONLY shows:
#   1. Prompt given to model
#   2. Model response
#   3. Latency
#
# No automatic JSON scoring.
# ============================================================

import json
import time
from datetime import datetime


# ============================================================
# 20 TEST CASES
# ============================================================

TESTS = [

    # ========================================================
    # CATEGORY 1 — VALID TRANSACTIONS (8)
    # ========================================================

    {
        "id": "valid_01",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract the financial transaction below.

Return ONLY valid JSON with exactly these fields:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Rules:
- amount, balance and fee must be numbers.
- type must be income, expense, or unknown.
- date must be YYYY-MM-DD.
- Do not invent missing information.
- Use null when information is unavailable.
- No markdown.
- No explanation.

SMS:
QGH7K3M2P1 Confirmed. You have received Ksh20,000.00 from Ann Mueni 0712***456 on 05/03/2026 at 10:42 AM. New M-PESA balance is Ksh159,583.00. Transaction cost, Ksh0.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_02",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this M-Pesa payment.

Return ONLY JSON with:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Do not invent missing information. Use null where necessary.

SMS:
TLA82K9P4Q Confirmed. Ksh1,250.00 paid to Naivas Supermarket on 06/03/2026 at 14:21. New M-PESA balance is Ksh158,333.00. Transaction cost, Ksh0.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_03",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this M-Pesa Till transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
RTP93LMN72 Confirmed. Ksh3,500.00 paid to 123456 - Wanga Electronics via M-PESA Till Number on 07/03/2026 at 09:15 AM. New M-PESA balance is Ksh154,833.00. Transaction cost, Ksh0.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_04",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this PayBill transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
PBY72LK91A Confirmed. Ksh5,000.00 sent to KPLC via PayBill 88888 for account 123456789 on 08/03/2026 at 18:03. New M-PESA balance is Ksh149,833.00. Transaction cost, Ksh0.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_05",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this bank transfer received.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
BNK72PQ91 Confirmed. Ksh45,000.00 received in your M-PESA account from Equity Bank on 09/03/2026 at 11:30 AM. New M-PESA balance is Ksh194,833.00. Reference EQT98431.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_06",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this Fuliza transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
FUL123ABC Confirmed. Fuliza loan of Ksh10,000.00 received on 10/03/2026 at 08:05 AM. New M-PESA balance is Ksh204,833.00. Fuliza outstanding balance is Ksh10,000.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_07",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this cash withdrawal.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
WD91KLM22 Confirmed. Ksh8,000.00 withdrawn from M-PESA at Agent 456789 - John Kamau on 11/03/2026 at 16:40. Transaction cost, Ksh80.00. New M-PESA balance is Ksh196,753.00.

Return ONLY JSON.
"""
    },

    {
        "id": "valid_08",
        "category": "valid_transaction",
        "task": "Extract transaction",
        "prompt": """
Extract this reversal.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

SMS:
REV8821 Confirmed. Reversal of Ksh2,500.00 for transaction QWE12345 has been credited to your M-PESA account on 12/03/2026 at 13:22. New M-PESA balance is Ksh199,253.00.

Return ONLY JSON.
"""
    },


    # ========================================================
    # CATEGORY 2 — MISSING / AMBIGUOUS DATA (6)
    # ========================================================

    {
        "id": "missing_01",
        "category": "missing_data",
        "task": "Handle missing balance",
        "prompt": """
Extract the transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Use null for information that is not present.
Do not guess.

SMS:
ABC12345 Confirmed. You have received Ksh7,500.00 from Mary Wanjiku on 13/03/2026 at 09:10 AM.

Return ONLY JSON.
"""
    },

    {
        "id": "missing_02",
        "category": "missing_data",
        "task": "Handle missing transaction ID",
        "prompt": """
Extract the transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Use null for missing information.

SMS:
You paid Ksh2,000.00 to Green Valley Shop on 14/03/2026 at 15:20. New M-PESA balance is Ksh197,253.00.

Return ONLY JSON.
"""
    },

    {
        "id": "missing_03",
        "category": "missing_data",
        "task": "Handle missing entity",
        "prompt": """
Extract the transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Do not invent an entity.

SMS:
TX99821 Confirmed. Ksh3,200.00 paid via M-PESA on 15/03/2026 at 12:00 PM. New M-PESA balance is Ksh194,053.00.

Return ONLY JSON.
"""
    },

    {
        "id": "missing_04",
        "category": "missing_data",
        "task": "Ambiguous transaction",
        "prompt": """
Determine what can safely be extracted from this message.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Use null when information is ambiguous or missing.
Do not invent facts.

SMS:
TX7712 Confirmed. Ksh5,000 sent. Balance Ksh100,000.

Return ONLY JSON.
"""
    },

    {
        "id": "missing_05",
        "category": "missing_data",
        "task": "Noisy SMS",
        "prompt": """
Extract the transaction from this noisy SMS.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Ignore irrelevant text.
Do not invent missing information.

SMS:
M-PESA ALERT!!! Your account was updated. TX88K21 Confirmed. You received Ksh12,000 from Peter on 16/03/2026. Please do not share your PIN. New balance Ksh112,000.

Return ONLY JSON.
"""
    },

    {
        "id": "missing_06",
        "category": "missing_data",
        "task": "Conflicting information",
        "prompt": """
Extract this transaction carefully.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

If the message contains conflicting information, preserve only what can be determined safely and use null where necessary.

SMS:
TX5566 Confirmed. Ksh4,000 paid to ABC Shop on 17/03/2026. New M-PESA balance is Ksh90,000. Later message says transaction amount was Ksh5,000.

Return ONLY JSON.
"""
    },


    # ========================================================
    # CATEGORY 3 — CAPABILITY / SELF-KNOWLEDGE (6)
    # ========================================================

    {
        "id": "capability_01",
        "category": "capability",
        "task": "Capabilities",
        "prompt": """
What types of financial transaction messages are you designed to understand?

Mention the transaction categories you can identify.

Answer concisely.
"""
    },

    {
        "id": "capability_02",
        "category": "capability",
        "task": "Supported fields",
        "prompt": """
What information can you extract from an M-Pesa or bank transaction message?

List the fields you are designed to identify.
"""
    },

    {
        "id": "capability_03",
        "category": "capability",
        "task": "Missing information",
        "prompt": """
What do you do when a financial SMS is missing important information such as the transaction ID, entity, balance, date, or amount?
"""
    },

    {
        "id": "capability_04",
        "category": "capability",
        "task": "Unsupported claims",
        "prompt": """
Can you access a user's bank account, M-Pesa account, contacts, internet, or private financial records directly?

Explain what you can and cannot access.
"""
    },

    {
        "id": "capability_05",
        "category": "capability",
        "task": "Role",
        "prompt": """
What is your role in the SME Ledger system?

Explain what happens after you extract a transaction from an SMS.
"""
    },

    {
        "id": "capability_06",
        "category": "capability",
        "task": "Transaction types",
        "prompt": """
Can you distinguish between income, expenses, transfers, withdrawals, merchant payments, PayBill payments, Fuliza transactions, reversals, and failed transactions?

If yes, briefly explain how.
"""
    }
]


# ============================================================
# RUN MODEL
# ============================================================

def run_model(prompt, max_tokens=256):

    start = time.time()

    try:

        response = llm.create_chat_completion(
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            top_p=1,
            seed=42,
            max_tokens=max_tokens
        )

        elapsed = time.time() - start

        text = response["choices"][0]["message"]["content"]

        return text.strip(), round(elapsed, 3), None

    except Exception as e:

        elapsed = time.time() - start

        return "", round(elapsed, 3), str(e)


# ============================================================
# RUN 20 TESTS
# ============================================================

results = []

print("\n")
print("=" * 90)
print("                 SME-LEDGER V2 — MANUAL 20-TEST EVALUATION")
print("=" * 90)


for i, test in enumerate(TESTS, start=1):

    print("\n\n")
    print("█" * 90)
    print(f"TEST {i}/20")
    print(f"ID       : {test['id']}")
    print(f"CATEGORY : {test['category']}")
    print(f"TASK     : {test['task']}")
    print("█" * 90)

    # --------------------------------------------------------
    # PROMPT
    # --------------------------------------------------------

    print("\n")
    print("┌" + "─" * 88 + "┐")
    print("│ PROMPT GIVEN TO MODEL")
    print("└" + "─" * 88 + "┘")

    print(test["prompt"])

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    response, latency, error = run_model(test["prompt"])

    print("\n")
    print("┌" + "─" * 88 + "┐")
    print("│ MODEL RESPONSE")
    print("└" + "─" * 88 + "┘")

    if error:
        print("❌ ERROR:")
        print(error)
    else:
        print(response)

    print("\n")
    print(f"⏱️  Latency: {latency:.3f} seconds")

    # --------------------------------------------------------
    # Store raw result
    # --------------------------------------------------------

    results.append({
        "test_number": i,
        "id": test["id"],
        "category": test["category"],
        "task": test["task"],
        "prompt": test["prompt"],
        "response": response,
        "latency_seconds": latency,
        "error": error
    })


# ============================================================
# SAVE RAW RESULTS
# ============================================================

OUTPUT_FILE = "sme_ledger_20_manual_test_results.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        {
            "evaluation": {
                "name": "SME-Ledger V2 Manual 20-Test Evaluation",
                "model": "sme-ledger-v2-Q4_K_M.gguf",
                "timestamp": datetime.utcnow().isoformat() + "Z",
                "seed": 42
            },
            "results": results
        },
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# FINAL RUN SUMMARY
# ============================================================

successful = sum(
    r["error"] is None
    for r in results
)

average_latency = sum(
    r["latency_seconds"]
    for r in results
) / len(results)


print("\n\n")
print("=" * 90)
print("                         TEST RUN COMPLETE")
print("=" * 90)

print(f"""
Tests executed:       {len(results)}/20
Successful runs:      {successful}/20
Average latency:      {average_latency:.3f} seconds

No automatic correctness or JSON validation was performed.
You are the evaluator.
""")

print(f"Raw results saved to: {OUTPUT_FILE}")
print("=" * 90)



                 SME-LEDGER V2 — MANUAL 20-TEST EVALUATION



██████████████████████████████████████████████████████████████████████████████████████████
TEST 1/20
ID       : valid_01
CATEGORY : valid_transaction
TASK     : Extract transaction
██████████████████████████████████████████████████████████████████████████████████████████


┌────────────────────────────────────────────────────────────────────────────────────────┐
│ PROMPT GIVEN TO MODEL
└────────────────────────────────────────────────────────────────────────────────────────┘

Extract the financial transaction below.

Return ONLY valid JSON with exactly these fields:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Rules:
- amount, balance and fee must be numbers.
- type must be income, expense, or unknown.
- date must be YYYY-MM-DD.
- Do not invent missing information.
- Use null when information is unavailable.
- No markdown.
- No explanation.

SMS:
QGH7K3M2P1 Confirmed. You have received

llama_perf_context_print:        load time =     793.09 ms
llama_perf_context_print: prompt eval time =     792.68 ms /   209 tokens (    3.79 ms per token,   263.66 tokens per second)
llama_perf_context_print:        eval time =    2383.37 ms /    93 runs   (   25.63 ms per token,    39.02 tokens per second)
llama_perf_context_print:       total time =    3308.15 ms /   302 tokens
llama_perf_context_print:    graphs reused =         91
Llama.generate: 5 prefix-match hit, remaining 136 prompt tokens to eval




┌────────────────────────────────────────────────────────────────────────────────────────┐
│ MODEL RESPONSE
└────────────────────────────────────────────────────────────────────────────────────────┘
transaction_id: QGH7K3M2P1
date: 2026-03-05
time: 10:42
type: Income
domain: bank_transfer_receive_money
entity: Ann Mueni
amount: 20000.0
balance: 0.0
fee: 0.0
reference: QGH7K3M2P1


⏱️  Latency: 3.315 seconds



██████████████████████████████████████████████████████████████████████████████████████████
TEST 2/20
ID       : valid_02
CATEGORY : valid_transaction
TASK     : Extract transaction
██████████████████████████████████████████████████████████████████████████████████████████


┌────────────────────────────────────────────────────────────────────────────────────────┐
│ PROMPT GIVEN TO MODEL
└────────────────────────────────────────────────────────────────────────────────────────┘

Extract this M-Pesa payment.

Return ONLY JSON with:
transaction_id, date, time, type, domain, entity, 

llama_perf_context_print:        load time =     793.09 ms
llama_perf_context_print: prompt eval time =     507.76 ms /   136 tokens (    3.73 ms per token,   267.84 tokens per second)
llama_perf_context_print:        eval time =    2460.79 ms /    95 runs   (   25.90 ms per token,    38.61 tokens per second)
llama_perf_context_print:       total time =    3107.27 ms /   231 tokens
llama_perf_context_print:    graphs reused =         94
Llama.generate: 10 prefix-match hit, remaining 133 prompt tokens to eval




┌────────────────────────────────────────────────────────────────────────────────────────┐
│ MODEL RESPONSE
└────────────────────────────────────────────────────────────────────────────────────────┘
transaction_id: TLA82K9P4Q
date: 2026-03-06
time: 14:21
type: Expense
domain: M-PESA
entity: Naivas Supermarket
amount: 1250.0
balance: 158333.0
fee: 0.0
reference: TLA82K9P4Q


⏱️  Latency: 3.112 seconds



██████████████████████████████████████████████████████████████████████████████████████████
TEST 3/20
ID       : valid_03
CATEGORY : valid_transaction
TASK     : Extract transaction
██████████████████████████████████████████████████████████████████████████████████████████


┌────────────────────────────────────────────────────────────────────────────────────────┐
│ PROMPT GIVEN TO MODEL
└────────────────────────────────────────────────────────────────────────────────────────┘

Extract this M-Pesa Till transaction.

Return ONLY JSON:
transaction_id, date, time, type, domain, entity, amo

...

llama_perf_context_print:        load time =     793.09 ms
llama_perf_context_print: prompt eval time =     118.59 ms /    28 tokens (    4.24 ms per token,   236.10 tokens per second)
llama_perf_context_print:        eval time =     946.53 ms /    36 runs   (   26.29 ms per token,    38.03 tokens per second)
llama_perf_context_print:       total time =    1118.65 ms /    64 tokens
llama_perf_context_print:    graphs reused =         35
Llama.generate: 4 prefix-match hit, remaining 42 prompt tokens to eval




┌────────────────────────────────────────────────────────────────────────────────────────┐
│ MODEL RESPONSE
└────────────────────────────────────────────────────────────────────────────────────────┘
I turn a M-PESA-Ledger-like assistant on-device, processing sensitive financial information and forwarding it to authorized financial intermediaries or cash-flow-oriented entities.


⏱️  Latency: 1.124 seconds



██████████████████████████████████████████████████████████████████████████████████████████
TEST 20/20
ID       : capability_06
CATEGORY : capability
TASK     : Transaction types
██████████████████████████████████████████████████████████████████████████████████████████


┌────────────────────────────────────────────────────────────────────────────────────────┐
│ PROMPT GIVEN TO MODEL
└────────────────────────────────────────────────────────────────────────────────────────┘

Can you distinguish between income, expenses, transfers, withdrawals, merchant payments, PayBill payments, F

llama_perf_context_print:        load time =     793.09 ms
llama_perf_context_print: prompt eval time =     173.06 ms /    42 tokens (    4.12 ms per token,   242.69 tokens per second)
llama_perf_context_print:        eval time =    1345.37 ms /    49 runs   (   27.46 ms per token,    36.42 tokens per second)
llama_perf_context_print:       total time =    1595.68 ms /    91 tokens
llama_perf_context_print:    graphs reused =         48




┌────────────────────────────────────────────────────────────────────────────────────────┐
│ MODEL RESPONSE
└────────────────────────────────────────────────────────────────────────────────────────┘
Yes. The application can analyze financial information in a supported manner to identify transaction or balance-related patterns, such as income, expenses, cash-flow or spending imbalances. This is a preliminary step and should be tailored to the specific application structure.


⏱️  Latency: 1.602 seconds



                         TEST RUN COMPLETE

Tests executed:       20/20
Successful runs:      20/20
Average latency:      2.789 seconds

No automatic correctness or JSON validation was performed.
You are the evaluator.

Raw results saved to: sme_ledger_20_manual_test_results.json


/tmp/ipykernel_16/1229522993.py:493: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",


In [5]:
# ============================================================
# SME-LEDGER V2 — INFERENCE ON test.csv
#
# Expected input: test.csv with a column containing transaction SMS.
# Output: predictions displayed below and saved to CSV.
# Uses the already-loaded `llm` from the model-loading cell above.
# ============================================================

import pandas as pd
import json
import time
from pathlib import Path
from IPython.display import display

TEST_CSV = "/kaggle/input/datasets/wangapa106g/massage-analysis/test_50_consistent.csv"
OUTPUT_CSV = "test_predictions.csv"

# 1) Load test data
test_df = pd.read_csv(TEST_CSV)
print(f"Loaded {len(test_df)} rows from {TEST_CSV}")
print("Columns:", test_df.columns.tolist())
display(test_df.head())

# 2) Identify the column containing messages.
# Set MESSAGE_COLUMN manually if your CSV uses a different column name.
MESSAGE_COLUMN = None  # e.g. "message" or "sms"

if MESSAGE_COLUMN is None:
    likely_names = [
        "message", "messages", "sms", "text", "transaction_message",
        "transaction", "raw_message"
    ]
    normalized = {str(c).strip().lower(): c for c in test_df.columns}
    MESSAGE_COLUMN = next(
        (normalized[name] for name in likely_names if name in normalized),
        None
    )

if MESSAGE_COLUMN is None:
    # If the file has exactly one column, treat it as the SMS column.
    if len(test_df.columns) == 1:
        MESSAGE_COLUMN = test_df.columns[0]
    else:
        raise ValueError(
            "Could not identify the SMS column. Set MESSAGE_COLUMN to the "
            "correct column name and rerun this cell. Available columns: "
            + ", ".join(map(str, test_df.columns))
        )

print(f"Using message column: {MESSAGE_COLUMN!r}")

# 3) Build the extraction prompt in the same field format used in this notebook.
def build_test_prompt(sms):
    return f"""Extract the financial transaction from the SMS below.

Return ONLY valid JSON with exactly these fields:
transaction_id, date, time, type, domain, entity, amount, balance, fee, reference

Rules:
- amount, balance and fee must be numbers or null.
- type must be income, expense, or unknown.
- date must be YYYY-MM-DD when available.
- Do not invent missing information; use null.
- No markdown and no explanation.

SMS:
{sms}

Return ONLY JSON."""

# 4) Run the already-loaded model on each message.
prediction_rows = []

for row_idx, sms in test_df[MESSAGE_COLUMN].items():
    if pd.isna(sms) or not str(sms).strip():
        prediction_rows.append({
            "source_row": row_idx,
            "sms": sms,
            "prediction": "",
            "latency_seconds": 0.0,
            "error": "Empty SMS"
        })
        continue

    prompt = build_test_prompt(str(sms))
    started = time.time()

    try:
        response = llm.create_chat_completion(
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            top_p=1,
            seed=42,
            max_tokens=256
        )
        prediction = response["choices"][0]["message"]["content"].strip()
        error = None
    except Exception as exc:
        prediction = ""
        error = str(exc)

    prediction_rows.append({
        "source_row": row_idx,
        "sms": str(sms),
        "prediction": prediction,
        "latency_seconds": round(time.time() - started, 3),
        "error": error
    })

predictions_df = pd.DataFrame(prediction_rows)

# 5) Show results and save them.
display(predictions_df)
predictions_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Saved {len(predictions_df)} predictions to: {OUTPUT_CSV}")

# Optional: parse JSON predictions into separate columns for easier inspection.
def parse_prediction(value):
    try:
        text = str(value).strip()
        # Accommodate accidental Markdown fences from model output.
        if text.startswith("```"):
            text = text.split("\\n", 1)[-1]
            if text.endswith("```"):
                text = text[:-3]
        parsed = json.loads(text)
        return parsed if isinstance(parsed, dict) else {"_parse_error": "JSON is not an object"}
    except Exception as exc:
        return {"_parse_error": str(exc)}

parsed_df = pd.json_normalize(predictions_df["prediction"].map(parse_prediction))
test_results = pd.concat(
    [predictions_df[["source_row", "sms", "latency_seconds", "error"]], parsed_df],
    axis=1
)

PARSED_OUTPUT_CSV = "test_predictions_parsed.csv"
test_results.to_csv(PARSED_OUTPUT_CSV, index=False, encoding="utf-8")
display(test_results.head(20))
print(f"Saved parsed predictions to: {PARSED_OUTPUT_CSV}")


Loaded 50 rows from /kaggle/input/datasets/wangapa106g/massage-analysis/test_50_consistent.csv
Columns: ['message']


,message
0,"TX03010001 Confirmed. You have received Ksh7,5..."
1,"TX03010002 Confirmed. You have received Ksh2,5..."
2,TX03020003 Confirmed. Ksh500.00 paid to Green ...
3,TX03020004 Confirmed. Ksh500.00 sent to David ...
4,"TX03030005 Confirmed. Ksh1,500.00 paid to Wang..."


Using message column: 'message'


Llama.generate: 4 prefix-match hit, remaining 194 prompt tokens to eval
llama_perf_context_print:        load time =     793.09 ms
llama_perf_context_print: prompt eval time =     729.71 ms /   194 tokens (    3.76 ms per token,   265.86 tokens per second)
llama_perf_context_print:        eval time =    2585.05 ms /    96 runs   (   26.93 ms per token,    37.14 tokens per second)
llama_perf_context_print:       total time =    3454.51 ms /   290 tokens
llama_perf_context_print:    graphs reused =         94
Llama.generate: 119 prefix-match hit, remaining 79 prompt tokens to eval
llama_perf_context_print:        load time =     793.09 ms
llama_perf_context_print: prompt eval time =     319.88 ms /    79 tokens (    4.05 ms per token,   246.97 tokens per second)
llama_perf_context_print:        eval time =    2658.87 ms /    99 runs   (   26.86 ms per token,    37.23 tokens per second)
llama_perf_context_print:       total time =    3128.19 ms /   178 tokens
llama_perf_context_print:    

,source_row,sms,prediction,latency_seconds,error
0,0,"TX03010001 Confirmed. You have received Ksh7,5...",transaction_id: TX03010001; date: 2026-03-01; ...,3.460,None
1,1,"TX03010002 Confirmed. You have received Ksh2,5...",transaction_id: TX03010002; date: 2026-03-01; ...,3.135,None
2,2,TX03020003 Confirmed. Ksh500.00 paid to Green ...,transaction_id: TX03020003,0.688,None
3,3,TX03020004 Confirmed. Ksh500.00 sent to David ...,transaction_id: TX03020004,0.697,None
4,4,"TX03030005 Confirmed. Ksh1,500.00 paid to Wang...",transaction_id: TX03030005; date: 2026-03-03; ...,3.079,None
5,5,"TX03030006 Confirmed. Ksh1,000.00 paid to Airt...",transaction_id: TX03030006,0.662,None
6,6,"TX03040007 Confirmed. You have received Ksh2,5...",transaction_id: TX03040007,0.692,None
7,7,TX03040008 Confirmed. Ksh500.00 sent to Peter ...,transaction_id: TX03040008; date: 2026-03-04; ...,2.931,None
8,8,"TX03050009 Confirmed. You have received Ksh7,5...",transaction_id: TX03050009,0.690,None
9,9,"TX03050010 Confirmed. You have received Ksh2,5...",transaction_id: TX03050010,0.671,None


Saved 50 predictions to: test_predictions.csv


,source_row,sms,latency_seconds,error,_parse_error
0,0,"TX03010001 Confirmed. You have received Ksh7,5...",3.460,None,Expecting value: line 1 column 1 (char 0)
1,1,"TX03010002 Confirmed. You have received Ksh2,5...",3.135,None,Expecting value: line 1 column 1 (char 0)
2,2,TX03020003 Confirmed. Ksh500.00 paid to Green ...,0.688,None,Expecting value: line 1 column 1 (char 0)
3,3,TX03020004 Confirmed. Ksh500.00 sent to David ...,0.697,None,Expecting value: line 1 column 1 (char 0)
4,4,"TX03030005 Confirmed. Ksh1,500.00 paid to Wang...",3.079,None,Expecting value: line 1 column 1 (char 0)
5,5,"TX03030006 Confirmed. Ksh1,000.00 paid to Airt...",0.662,None,Expecting value: line 1 column 1 (char 0)
6,6,"TX03040007 Confirmed. You have received Ksh2,5...",0.692,None,Expecting value: line 1 column 1 (char 0)
7,7,TX03040008 Confirmed. Ksh500.00 sent to Peter ...,2.931,None,Expecting value: line 1 column 1 (char 0)
8,8,"TX03050009 Confirmed. You have received Ksh7,5...",0.690,None,Expecting value: line 1 column 1 (char 0)
9,9,"TX03050010 Confirmed. You have received Ksh2,5...",0.671,None,Expecting value: line 1 column 1 (char 0)


Saved parsed predictions to: test_predictions_parsed.csv


## Financial analysis and dashboard
Run this cell after the `test.csv` inference cell. It builds a Pandas ledger, computes financial KPIs, displays charts and filters, and exports analysis CSVs.

In [6]:
# ============================================================
# SME-LEDGER V2 — PANDAS ANALYTICS & DASHBOARD
# Run AFTER the test.csv inference cell above.
# Builds a normalized ledger from model JSON, computes KPIs,
# and displays interactive Plotly charts + a transaction table.
# ============================================================

import json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Use parsed test_results if the inference cell created it.
# Otherwise parse predictions_df directly.
if "test_results" in globals():
    raw_results = test_results.copy()
elif "predictions_df" in globals():
    raw_results = predictions_df.copy()
else:
    raise NameError("Run the test.csv inference cell first.")

def _safe_json(value):
    if isinstance(value, dict):
        return value
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return {}
    # Strip possible fenced code blocks
    if text.startswith("```"):
        text = text.split("\n", 1)[-1]
        if text.endswith("```"):
            text = text[:-3]
    try:
        obj = json.loads(text)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}

# Normalize one model response per source row.
prediction_col = "prediction" if "prediction" in raw_results.columns else None
if prediction_col:
    extracted = raw_results[prediction_col].apply(_safe_json).apply(pd.Series)
    ledger = pd.concat(
        [raw_results.reset_index(drop=True), extracted.reset_index(drop=True)],
        axis=1
    )
else:
    # test_results already contains parsed fields
    ledger = raw_results.copy()

# Keep original SMS and normalize expected fields.
field_defaults = {
    "type": "unknown", "domain": "unknown", "entity": "Unknown",
    "amount": np.nan, "balance": np.nan, "fee": np.nan,
    "date": None, "time": None, "reference": None, "transaction_id": None
}
for col, default in field_defaults.items():
    if col not in ledger.columns:
        ledger[col] = default

def _num(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False)
                    .str.replace("Ksh", "", regex=False)
                    .str.replace("KES", "", regex=False)
                    .str.strip()
                    .replace({"": np.nan, "None": np.nan, "null": np.nan, "nan": np.nan}),
        errors="coerce"
    )

for col in ["amount", "balance", "fee"]:
    ledger[col] = _num(ledger[col])

ledger["type"] = ledger["type"].astype(str).str.strip().str.lower()
ledger["direction"] = ledger["type"].map({
    "income": "Income", "received": "Income", "deposit": "Income",
    "expense": "Expense", "payment": "Expense", "sent": "Expense",
    "withdrawal": "Expense"
}).fillna("Unknown")

ledger["entity"] = ledger["entity"].fillna("Unknown").astype(str).replace({"": "Unknown", "nan": "Unknown"})
ledger["domain"] = ledger["domain"].fillna("Unknown").astype(str).replace({"": "Unknown", "nan": "Unknown"})
ledger["date_parsed"] = pd.to_datetime(ledger["date"], errors="coerce", dayfirst=True)
ledger["month"] = ledger["date_parsed"].dt.to_period("M").astype(str)
ledger.loc[ledger["date_parsed"].isna(), "month"] = "Unknown date"

# Amounts are signed based on the model's extracted transaction type.
ledger["income_kes"] = np.where(ledger["direction"].eq("Income"), ledger["amount"], 0)
ledger["expense_kes"] = np.where(ledger["direction"].eq("Expense"), ledger["amount"], 0)
ledger["net_cashflow_kes"] = ledger["income_kes"] - ledger["expense_kes"]

# Basic data quality indicators. These are flags for review, not proof of error.
ledger["missing_amount"] = ledger["amount"].isna()
ledger["unknown_direction"] = ledger["direction"].eq("Unknown")
ledger["missing_date"] = ledger["date_parsed"].isna()
if "reference" in ledger.columns:
    ref = ledger["reference"].astype(str).str.strip()
    ledger["possible_duplicate"] = ref.ne("") & ref.ne("nan") & ref.duplicated(keep=False)
else:
    ledger["possible_duplicate"] = False

# KPI values
n_tx = len(ledger)
known_amount = ledger["amount"].notna()
total_income = ledger["income_kes"].sum()
total_expenses = ledger["expense_kes"].sum()
net_cashflow = total_income - total_expenses
avg_tx = ledger.loc[known_amount, "amount"].mean()
latest_balance = ledger.loc[ledger["balance"].notna(), "balance"].iloc[-1] if ledger["balance"].notna().any() else np.nan
income_expense_ratio = total_income / total_expenses if total_expenses else np.nan

display(Markdown("# SME-Ledger Test Data — Financial Dashboard"))
display(Markdown(
    f"**Rows:** {n_tx:,} &nbsp; | &nbsp; "
    f"**Income:** KES {total_income:,.2f} &nbsp; | &nbsp; "
    f"**Expenses:** KES {total_expenses:,.2f} &nbsp; | &nbsp; "
    f"**Net cash flow:** KES {net_cashflow:,.2f}"
))

# KPI cards as a compact dataframe (renders cleanly in Jupyter/Kaggle)
kpis = pd.DataFrame({
    "Metric": ["Transactions", "Total income (KES)", "Total expenses (KES)",
               "Net cash flow (KES)", "Average transaction (KES)",
               "Latest extracted balance (KES)", "Income / expense ratio"],
    "Value": [
        f"{n_tx:,}", f"{total_income:,.2f}", f"{total_expenses:,.2f}",
        f"{net_cashflow:,.2f}",
        f"{avg_tx:,.2f}" if pd.notna(avg_tx) else "N/A",
        f"{latest_balance:,.2f}" if pd.notna(latest_balance) else "N/A",
        f"{income_expense_ratio:.2f}" if pd.notna(income_expense_ratio) else "N/A"
    ]
})
display(kpis)

# Filters (interactive widgets when ipywidgets is available)
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    entity_options = ["All"] + sorted(ledger["entity"].dropna().astype(str).unique().tolist())
    direction_options = ["All"] + sorted(ledger["direction"].dropna().astype(str).unique().tolist())
    entity_filter = widgets.Dropdown(options=entity_options, value="All", description="Entity:")
    direction_filter = widgets.Dropdown(options=direction_options, value="All", description="Flow:")
    output_area = widgets.Output()

    def _render_dashboard(*_):
        with output_area:
            clear_output(wait=True)
            view = ledger.copy()
            if entity_filter.value != "All":
                view = view[view["entity"].astype(str) == entity_filter.value]
            if direction_filter.value != "All":
                view = view[view["direction"].astype(str) == direction_filter.value]
            display(Markdown(f"### Filtered transactions: {len(view):,}"))
            display(view[["sms", "date", "type", "entity", "amount", "balance", "fee", "reference", "direction"]]
                    .head(100) if "sms" in view.columns else view.head(100))
            _plot_dashboard(view)

    def _plot_dashboard(view):
        # Monthly income vs expenses
        monthly = view.groupby("month", dropna=False)[["income_kes", "expense_kes"]].sum().reset_index()
        monthly = monthly[monthly["month"] != "Unknown date"]
        if not monthly.empty:
            long_monthly = monthly.melt(id_vars="month", var_name="Cash flow", value_name="KES")
            fig = px.bar(long_monthly, x="month", y="KES", color="Cash flow",
                         barmode="group", title="Monthly Income vs Expenses")
            fig.update_layout(xaxis_title="Month", yaxis_title="KES")
            fig.show()

        # Spending by category/domain
        spend = view[view["direction"] == "Expense"].groupby("domain", dropna=False)["amount"].sum().reset_index()
        spend = spend[spend["amount"] > 0].sort_values("amount", ascending=False)
        if not spend.empty:
            fig = px.pie(spend, names="domain", values="amount", hole=0.45,
                         title="Expenses by Extracted Domain / Category")
            fig.show()

        # Top merchants/entities
        merchants = view[view["direction"] == "Expense"].groupby("entity")["amount"].sum().nlargest(10).reset_index()
        if not merchants.empty:
            fig = px.bar(merchants.sort_values("amount"), x="amount", y="entity",
                         orientation="h", title="Top 10 Expense Entities",
                         labels={"amount": "KES", "entity": "Entity"})
            fig.show()

        # Transaction distribution
        counts = view["direction"].value_counts().rename_axis("Direction").reset_index(name="Transactions")
        if not counts.empty:
            fig = px.bar(counts, x="Direction", y="Transactions", title="Transaction Counts by Direction")
            fig.show()

        # Balance trend, only when valid dates and balances exist
        balances = view.dropna(subset=["date_parsed", "balance"]).sort_values("date_parsed")
        if not balances.empty:
            fig = px.line(balances, x="date_parsed", y="balance", markers=True,
                          title="Extracted Balance Over Time",
                          labels={"date_parsed": "Date", "balance": "Balance (KES)"})
            fig.show()

        # Data quality
        quality = pd.DataFrame({
            "Check": ["Missing amount", "Unknown direction", "Missing/unparsed date", "Possible duplicate reference"],
            "Rows": [
                int(view["missing_amount"].sum()), int(view["unknown_direction"].sum()),
                int(view["missing_date"].sum()), int(view["possible_duplicate"].sum())
            ]
        })
        fig = px.bar(quality, x="Check", y="Rows", title="Extraction Quality Checks")
        fig.show()

    entity_filter.observe(_render_dashboard, names="value")
    direction_filter.observe(_render_dashboard, names="value")
    display(widgets.HBox([entity_filter, direction_filter]))
    display(output_area)
    _render_dashboard()
except ImportError:
    # Static charts if ipywidgets is unavailable
    _plot_dashboard(ledger)

# Full normalized ledger and downloadable outputs
display(Markdown("## Normalized transaction ledger"))
display(ledger.head(100))

LEDGER_CSV = "sme_ledger_test_analytics.csv"
SUMMARY_CSV = "sme_ledger_test_monthly_summary.csv"
ledger.to_csv(LEDGER_CSV, index=False)
monthly_summary = ledger.groupby("month", dropna=False)[
    ["income_kes", "expense_kes", "net_cashflow_kes"]
].sum().reset_index()
monthly_summary.to_csv(SUMMARY_CSV, index=False)

print(f"Saved normalized ledger: {LEDGER_CSV}")
print(f"Saved monthly summary: {SUMMARY_CSV}")


/tmp/ipykernel_16/600078871.py:70: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"": np.nan, "None": np.nan, "null": np.nan, "nan": np.nan}),
/tmp/ipykernel_16/600078871.py:70: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"": np.nan, "None": np.nan, "null": np.nan, "nan": np.nan}),
/tmp/ipykernel_16/600078871.py:70: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to 

# SME-Ledger Test Data — Financial Dashboard

**Rows:** 50 &nbsp; | &nbsp; **Income:** KES 0.00 &nbsp; | &nbsp; **Expenses:** KES 0.00 &nbsp; | &nbsp; **Net cash flow:** KES 0.00

,Metric,Value
0,Transactions,50
1,Total income (KES),0.00
2,Total expenses (KES),0.00
3,Net cash flow (KES),0.00
4,Average transaction (KES),N/A
5,Latest extracted balance (KES),N/A
6,Income / expense ratio,N/A


Output()

## Normalized transaction ledger

,source_row,sms,latency_seconds,error,_parse_error,type,domain,entity,amount,balance,...,direction,date_parsed,month,income_kes,expense_kes,net_cashflow_kes,missing_amount,unknown_direction,missing_date,possible_duplicate
0,0,"TX03010001 Confirmed. You have received Ksh7,5...",3.460,None,Expecting value: line 1 column 1 (char 0),unknown,unknown,Unknown,NaN,NaN,...,Unknown,NaT,Unknown date,0.0,0.0,0.0,True,True,True,True
1,1,"TX03010002 Confirmed. You have received Ksh2,5...",3.135,None,Expecting value: line 1 column 1 (char 0),unknown,unknown,Unknown,NaN,NaN,...,Unknown,NaT,Unknown date,0.0,0.0,0.0,True,True,True,True
2,2,TX03020003 Confirmed. Ksh500.00 paid to Green ...,0.688,None,Expecting value: line 1 column 1 (char 0),unknown,unknown,Unknown,NaN,NaN,...,Unknown,NaT,Unknown date,0.0,0.0,0.0,True,True,True,True
3,3,TX03020004 Confirmed. Ksh500.00 sent to David ...,0.697,None,Expecting value: line 1 column 1 (char 0),unknown,unknown,Unknown,NaN,NaN,...,Unknown,NaT,Unknown date,0.0,0.0,0.0,True,True,True,True
4,4,"TX03030005 Confirmed. Ksh1,500.00 paid to Wang...",3.079,None,Expecting value: line 1 column 1 (char 0),unknown,unknown,Unknown,NaN,NaN,...,Unknown,NaT,Unknown date,0.0,0.0,0.0,True,True,True,True
5,5,"TX03030006 Confirmed. Ksh1,000.00 paid to Airt...",0.662,None,Expecting value: line 1 column 1 (char 0),unknown,unknown,Unknown,NaN,NaN,...,Unknown,NaT,Unknown date,0.0,0.0,0.0,True,True,True,True
6,6,"TX03040007 Confirmed. You have received Ksh2,5...",0.692,None,Expecting value: line 1 column 1 (char 0),unknown,unknown,Unknown,NaN,NaN,...,Unknown,NaT,Unknown date,0.0,0.0,0.0,True,True,True,True
7,7,TX03040008 Confirmed. Ksh500.00 sent to Peter ...,2.931,None,Expecting value: line 1 column 1 (char 0),unknown,unknown,Unknown,NaN,NaN,...,Unknown,NaT,Unknown date,0.0,0.0,0.0,True,True,True,True
8,8,"TX03050009 Confirmed. You have received Ksh7,5...",0.690,None,Expecting value: line 1 column 1 (char 0),unknown,unknown,Unknown,NaN,NaN,...,Unknown,NaT,Unknown date,0.0,0.0,0.0,True,True,True,True
9,9,"TX03050010 Confirmed. You have received Ksh2,5...",0.671,None,Expecting value: line 1 column 1 (char 0),unknown,unknown,Unknown,NaN,NaN,...,Unknown,NaT,Unknown date,0.0,0.0,0.0,True,True,True,True


Saved normalized ledger: sme_ledger_test_analytics.csv
Saved monthly summary: sme_ledger_test_monthly_summary.csv
